In [ ]:
import carla
import numpy as np
import cv2
import queue
from ultralytics import YOLO

import typing
import collections

# Python 3.7의 typing 모듈에 없는 OrderedDict를 강제로 연결
if not hasattr(typing, 'OrderedDict'):
    typing.OrderedDict = collections.OrderedDict





def main():
    # 1. 서버 연결 및 동기 모드(Synchronous Mode) 강제화
    client = carla.Client('localhost', 2000)
    client.set_timeout(10.0)
    world = client.get_world()
    
    settings = world.get_settings()
    settings.synchronous_mode = True
    settings.fixed_delta_seconds = 0.05  # 20 FPS로 고정 (물리 엔진 틱 간격)
    world.apply_settings(settings)

    actor_list = []

    try:
        # 2. 차량 스폰 (Ego Vehicle)
        blueprint_library = world.get_blueprint_library()
        vehicle_bp = blueprint_library.filter('vehicle.tesla.model3')[0]
        spawn_point = world.get_map().get_spawn_points()[0]
        vehicle = world.spawn_actor(vehicle_bp, spawn_point)
        actor_list.append(vehicle)

        # 3. 카메라 센서 부착 및 설정
        camera_bp = blueprint_library.find('sensor.camera.rgb')
        camera_bp.set_attribute('image_size_x', '640') # YOLO 입력 크기에 맞게 최적화
        camera_bp.set_attribute('image_size_y', '640')
        camera_bp.set_attribute('fov', '90')
        
        # 차량의 전면 유리창 부근으로 트랜스폼 설정
        camera_transform = carla.Transform(carla.Location(x=1.5, z=2.4))
        camera = world.spawn_actor(camera_bp, camera_transform, attach_to=vehicle)
        actor_list.append(camera)

        # 4. 데이터 수집 큐(Queue) 설정
        image_queue = queue.Queue()
        camera.listen(image_queue.put)

        

        # 5. YOLOv8 모델 로드
        model = YOLO('/home/oem/workspace/carla_workshop/YOLO/runs/detect/train2/weights/best.pt') # coco8 학습 모델


        print("동기 모드 기반 추론 파이프라인 가동 시작...")

        # 6. 메인 제어 루프
        while True:
            # 6-1. 서버 틱 전진 (이 명령이 없으면 서버는 대기함)
            world.tick()

            # 6-2. 큐에서 틱과 동기화된 센서 데이터 획득
            image = image_queue.get()

            # 6-3. 원시 데이터(Raw Data)를 NumPy 배열로 변환 (치명적 병목 해결 지점)
            # CARLA 이미지는 BGRA 포맷의 1차원 배열로 전달됨
            img_array = np.frombuffer(image.raw_data, dtype=np.dtype("uint8"))
            img_array = np.reshape(img_array, (image.height, image.width, 4))
            
            # Alpha 채널 제거 (BGRA -> BGR) : YOLO는 3채널을 요구함
            frame_bgr = img_array[:, :, :3]

            # 6-4. YOLOv8 추론 실행
            results = model.predict(source=frame_bgr, verbose=False)

            # 6-5. 행동 결정 (Action Determination) 및 제어 명령 전달
            control = carla.VehicleControl()
            control.throttle = 0.5 # 기본 직진 가속
            
            # 예시: 사람(class 0)이 탐지되면 브레이크 작동
            for r in results:
                boxes = r.boxes
                for box in boxes:
                    cls = int(box.cls[0])
                    if cls == 0: 
                        control.throttle = 0.0
                        control.brake = 1.0
                        break # 긴급 제동

            vehicle.apply_control(control)

            # (선택) 시각화를 위한 화면 출력 (속도 저하의 원인이 될 수 있으므로 테스트용으로만 사용)
            annotated_frame = results[0].plot()
            cv2.imshow("YOLOv8 CARLA Inference", annotated_frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                 break

    finally:
        # 7. 종료 시 리소스 정리 및 비동기 모드 복구 (필수)
        print("리소스 정리 중...")
        settings.synchronous_mode = False
        world.apply_settings(settings)
        for actor in actor_list:
            actor.destroy()
        cv2.destroyAllWindows()

if __name__ == '__main__':
    main()